# 03 — Fine-tune PP-OCRv5 reader

    Mở runtime GPU mới. Dataset recognizer đã gồm crop từ ảnh camera gốc và vùng zoom; chỉ dùng crop có nhãn được duyệt. Notebook lấy source PaddleOCR và training weights chính thức, tạo config dựa trên upstream, lưu checkpoint trên Drive. Kiểm tra wheel PaddlePaddle hợp với CUDA/driver thực tế trước khi chạy cell cài đặt.


In [2]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
from google.colab import drive

REPO_URL = 'https://github.com/huybitvvt/scale-ocr.git'
PROJECT_COMMIT = 'b0902866d0a04a015a9b0edf2f75f106d1851ef1'  # Code cố định cho dataset v001.
GITHUB_SECRET_NAME = ''  # Repo public không cần token; chỉ đặt tên secret nếu chuyển private.
VERSION = 'v001'
DRIVE_ROOT = Path('/content/drive/MyDrive/tram-can')
LOCAL_ROOT = Path('/content/scale-data')
REPO_ROOT = Path('/content/scale-ocr-code')
drive._mount('/content/drive')
assert DRIVE_ROOT.is_dir(), 'Không thấy MyDrive/tram-can: mount đúng tài khoản Google chứa ZIP'
assert REPO_URL.startswith(('https://', 'git@')), 'Điền REPO_URL sau khi push Git'
if not REPO_ROOT.exists():
    git_env = os.environ.copy()
    askpass = Path('/content/scale-ocr-askpass.sh')
    try:
        if GITHUB_SECRET_NAME:
            from google.colab import userdata
            assert REPO_URL.startswith('https://github.com/'), 'Secret này chỉ dùng với github.com'
            token = userdata.get(GITHUB_SECRET_NAME)
            assert token, 'Chưa có Colab Secret GITHUB_TOKEN hoặc chưa cấp quyền notebook đọc secret'
            git_env['SCALE_GIT_TOKEN'] = token
            git_env['GIT_TERMINAL_PROMPT'] = '0'
            git_env['GIT_ASKPASS'] = str(askpass)
            askpass.write_text(chr(10).join([
                '#!/bin/sh', 'case "$1" in',
                '  *Username*) printf "%s\n" "x-access-token" ;;',
                '  *) printf "%s\n" "$SCALE_GIT_TOKEN" ;;',
                'esac', ''
            ]), encoding='utf-8')
            askpass.chmod(0o700)
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)],
                       env=git_env, check=True)
    finally:
        git_env.pop('SCALE_GIT_TOKEN', None)
        askpass.unlink(missing_ok=True)
if PROJECT_COMMIT:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', PROJECT_COMMIT], check=True)
PROJECT_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT)], check=True)
print('Code commit:', PROJECT_COMMIT)
print('Disk GiB free:', round(shutil.disk_usage('/content').free / 2**30, 2))
subprocess.run(['nvidia-smi'], check=False)


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Dataset đã duyệt nằm trên Drive; vòng lặp train chỉ đọc từ /content.
bundle = DRIVE_ROOT / 'datasets' / VERSION / 'dataset.zip'
checksum_file = bundle.with_name(bundle.name + '.sha256')
assert bundle.is_file() and checksum_file.is_file(), 'Upload dataset.zip và .sha256 sau khi gán/duyệt nhãn'
expected = checksum_file.read_text(encoding='ascii').split()[0].lower()
assert expected == '0eebc32cb89e8d264d0450292e57aeeb81388efe0d9d15848ab3d7489b91870b', 'Sai checksum dataset v001'
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
local_bundle = LOCAL_ROOT / bundle.name
def sha256_stream(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
if not local_bundle.exists() or sha256_stream(local_bundle) != expected:
    shutil.copyfile(bundle, local_bundle)
actual = sha256_stream(local_bundle)
assert actual == expected, f'Dataset SHA-256 mismatch: {actual}'
labels_zip = DRIVE_ROOT / 'labels' / VERSION / 'labels_reviewed_v001.zip'
labels_checksum = labels_zip.with_name(labels_zip.name + '.sha256')
assert labels_zip.is_file() and labels_checksum.is_file(), f'Missing reviewed labels: {labels_zip}'
labels_expected = 'c2f0a77dacad33face7c74a5180fce5ae79498514e38d89730de4e22c94f78b6'
assert labels_checksum.read_text(encoding='ascii').split()[0].lower() == labels_expected
assert sha256_stream(labels_zip) == labels_expected, 'Sai SHA-256 labels_reviewed_v001.zip'
print('Reviewed labels SHA-256:', labels_expected)
prepared = LOCAL_ROOT / 'prepared' / VERSION
if not (prepared / 'dataset_card.json').is_file():
    prepared.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_bundle) as archive:
        for item in archive.infolist():
            target = (prepared / item.filename).resolve()
            assert target.is_relative_to(prepared.resolve()), 'Unsafe ZIP path'
        archive.extractall(prepared)
card = json.loads((prepared / 'dataset_card.json').read_text(encoding='utf-8'))
assert card['counts'].get('recognizer_crops_train', 0) > 0, 'Dataset thiếu crop train'
print('Dataset SHA-256:', actual)
print('Counts:', card['counts'])


In [ ]:
# Ví dụ wheel CUDA 12.6 theo hướng dẫn Paddle; sửa theo runtime hiện được cấp.
PADDLE_WHEEL_INDEX = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
PADDLE_WHEEL = 'paddlepaddle-gpu==3.3.0'
subprocess.run([sys.executable, '-m', 'pip', 'install', PADDLE_WHEEL,
                '-i', PADDLE_WHEEL_INDEX], check=True)
PADDLE_ROOT = Path('/content/PaddleOCR')
PADDLE_REF = 'release/3.5'  # Sau smoke test, thay bằng commit SHA cố định.
if not PADDLE_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', PADDLE_REF,
        'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLE_ROOT)], check=True)
paddle_commit = subprocess.check_output(
    ['git', '-C', str(PADDLE_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(PADDLE_ROOT / 'requirements.txt')], check=True)
import paddle, yaml
assert paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
paddle.set_device('gpu:0')
paddle.utils.run_check()
print('PaddleOCR commit:', paddle_commit)


In [ ]:
from urllib.request import urlretrieve
from datetime import datetime, timezone
pretrained = LOCAL_ROOT / 'pretrained' / 'PP-OCRv5_server_rec_pretrained.pdparams'
pretrained.parent.mkdir(parents=True, exist_ok=True)
if not pretrained.exists():
    urlretrieve('https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/'
                'PP-OCRv5_server_rec_pretrained.pdparams', pretrained)
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
rec_root = prepared / 'recognizer'
for split in ('train', 'val'):
    assert (rec_root / (split + '.txt')).stat().st_size > 0, f'Thiếu nhãn {split}'
base_config = PADDLE_ROOT / 'configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml'
cfg = yaml.safe_load(base_config.read_text(encoding='utf-8'))
REC_RUN_ID = 'rec-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
rec_run = DRIVE_ROOT / 'runs' / REC_RUN_ID
rec_run.mkdir(parents=True, exist_ok=False)
cfg['Global'].update({'use_gpu': True, 'distributed': False, 'epoch_num': 50,
    'pretrained_model': str(pretrained), 'checkpoints': None,
    'save_model_dir': str(rec_run / 'checkpoints'), 'save_epoch_step': 5,
    'eval_batch_step': [0, 100]})
cfg['Optimizer']['lr']['learning_rate'] = 0.0001
for section, label in [('Train', 'train.txt'), ('Eval', 'val.txt')]:
    cfg[section]['dataset']['data_dir'] = str(rec_root)
    cfg[section]['dataset']['label_file_list'] = [str(rec_root / label)]
    cfg[section]['loader']['num_workers'] = 2
    cfg[section]['loader']['batch_size_per_card'] = 16
cfg['Train']['sampler']['first_bs'] = 16
cfg['Train']['sampler']['fix_bs'] = True
cfg['Train']['sampler']['scales'] = [[320, 48]]
cfg['Train']['dataset']['transforms'] = [t for t in cfg['Train']['dataset']['transforms']
    if 'RecAug' not in t]
rec_config = rec_run / 'recognizer.yaml'
rec_config.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
(rec_run / 'run_manifest.json').write_text(json.dumps({
    'project_commit': PROJECT_COMMIT, 'dataset_sha256': actual,
    'dataset_version': VERSION, 'paddleocr_commit': paddle_commit,
    'pretrained_sha256': sha256(pretrained), 'paddle': paddle.__version__,
    'config_sha256': sha256(rec_config), 'seed': 42,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Run:', REC_RUN_ID)


## Smoke test trước run chính

    Cell này train 1 epoch trên 64 crop train và 16 crop val. Checkpoint thử nghiệm nằm trong `/content`; chỉ chạy cell train chính nếu smoke test kết thúc thành công và log cho thấy model nạp pretrained đúng.


In [ ]:
import copy

smoke_dir = LOCAL_ROOT / 'smoke' / REC_RUN_ID
smoke_dir.mkdir(parents=True, exist_ok=True)
train_lines = (rec_root / 'train.txt').read_text(encoding='utf-8').splitlines(keepends=True)
val_lines = (rec_root / 'val.txt').read_text(encoding='utf-8').splitlines(keepends=True)
assert len(train_lines) >= 64 and len(val_lines) >= 16
smoke_train = smoke_dir / 'train.txt'
smoke_val = smoke_dir / 'val.txt'
smoke_train.write_text(''.join(train_lines[:64]), encoding='utf-8')
smoke_val.write_text(''.join(val_lines[:16]), encoding='utf-8')
smoke_cfg = copy.deepcopy(cfg)
smoke_cfg['Global'].update({
    'epoch_num': 1, 'save_epoch_step': 1,
    'eval_batch_step': [0, 4],
    'save_model_dir': str(smoke_dir / 'checkpoints'),
})
smoke_cfg['Train']['dataset']['label_file_list'] = [str(smoke_train)]
smoke_cfg['Eval']['dataset']['label_file_list'] = [str(smoke_val)]
smoke_cfg['Train']['sampler']['first_bs'] = 8
smoke_cfg['Train']['loader']['batch_size_per_card'] = 8
smoke_cfg['Eval']['loader']['batch_size_per_card'] = 8
smoke_config = smoke_dir / 'recognizer_smoke.yaml'
smoke_config.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False), encoding='utf-8')
subprocess.run([sys.executable, 'tools/train.py', '-c', str(smoke_config)],
               cwd=PADDLE_ROOT, check=True)
print('Smoke test completed:', smoke_dir)


## Train reader trên toàn bộ tập train

    Chạy cell này sau khi smoke test thành công. Checkpoint của run chính ghi vào Drive.


In [ ]:
subprocess.run([sys.executable, 'tools/train.py', '-c', str(rec_config)],
               cwd=PADDLE_ROOT, check=True)


## Resume khi Colab ngắt

    Chạy lại bootstrap, copy cùng dataset, cài PaddleOCR rồi chọn prefix checkpoint cũ có cả `.pdparams` và `.pdopt`. Giữ nguyên config cũ; `Global.checkpoints` phục hồi trạng thái train.


In [ ]:
RESUME_RUN_ID = ''
CHECKPOINT_NAME = ''  # Ví dụ: latest hoặc epoch_10, theo tên thực tế trong Drive.
if RESUME_RUN_ID:
    old_run = DRIVE_ROOT / 'runs' / RESUME_RUN_ID
    old = json.loads((old_run / 'run_manifest.json').read_text(encoding='utf-8'))
    assert old['project_commit'] == PROJECT_COMMIT and old['dataset_sha256'] == actual
    old_cfg = old_run / 'recognizer.yaml'
    assert sha256(old_cfg) == old['config_sha256']
    prefix = old_run / 'checkpoints' / CHECKPOINT_NAME
    assert prefix.with_suffix('.pdparams').is_file()
    assert prefix.with_suffix('.pdopt').is_file()
    subprocess.run([sys.executable, 'tools/train.py', '-c', str(old_cfg),
        '-o', 'Global.checkpoints=' + str(prefix)], cwd=PADDLE_ROOT, check=True)
